# Caso 6 — Análise por Consulta

Selecionamos, a partir das diferenças de AP calculadas no Caso 5:

- **2 consultas em que o BM25 é claramente superior** ao Modelo Vetorial;
- **2 consultas em que o Modelo Vetorial é claramente superior** ao BM25;
- **2 consultas em que ambos os modelos têm desempenho insatisfatório**.

Para cada uma, mostramos os 5 primeiros documentos retornados por cada
modelo, indicando quais são de fato relevantes segundo o qrels.


In [1]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pickle
import pandas as pd
from IPython.display import display

from src.cranfield_data import load_cranfield

df_docs, df_queries, df_qrels = load_cranfield()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))
query_text = dict(zip(df_queries.query_id, df_queries.text.str.replace("\n", " ")))

RESULTS_DIR = project_root / "data" / "processed"
vsm_eval = pd.read_csv(RESULTS_DIR / "vsm_perquery.csv", dtype={"query_id": str}).set_index("query_id")
bm25_eval = pd.read_csv(RESULTS_DIR / "bm25_perquery.csv", dtype={"query_id": str}).set_index("query_id")
with open(RESULTS_DIR / "vsm_rankings.pkl", "rb") as f:
    vsm_rankings = pickle.load(f)
with open(RESULTS_DIR / "bm25_rankings.pkl", "rb") as f:
    bm25_rankings = pickle.load(f)


def show_top_n(qid, n=5):
    """Mostra os top-n documentos de cada modelo para uma consulta, com o
    grau de relevância (qrels) de cada documento retornado."""
    grades = dict(zip(df_qrels[df_qrels.query_id == qid].doc_id,
                       df_qrels[df_qrels.query_id == qid].relevance))
    print(f"Consulta {qid}: {query_text[qid]}")
    print(f"AP -> BM25: {bm25_eval.loc[qid, 'AP']:.3f} | "
          f"Modelo Vetorial: {vsm_eval.loc[qid, 'AP']:.3f}")
    frames = {}
    for nome, rankings in [("BM25", bm25_rankings), ("Modelo Vetorial", vsm_rankings)]:
        rows = []
        for rank, did in enumerate(rankings[qid][:n], start=1):
            grade = grades.get(did)
            rows.append({
                "rank": rank,
                "doc_id": did,
                "grau": grade if grade is not None else "não julgado",
                "relevante?": "SIM" if (grade is not None and grade >= 1) else "não",
                "titulo": doc_title[did][:75],
            })
        frames[nome] = pd.DataFrame(rows).set_index("rank")
    for nome, frame in frames.items():
        print(f"-- Top-{n} {nome} --")
        display(frame)
    print()

## 1) Consultas em que o BM25 é claramente superior

**Consulta 167** (AP 0,750 no BM25 contra 0,091 no Vetorial) tem dois
documentos relevantes: o 274, de grau 2, e o 82, de grau 3. O Modelo
Vetorial não traz nenhum dos dois ao Top 5 — todo o seu topo é ocupado por
documentos não julgados que repetem `ablat` de forma concentrada, como o
553 (8 ocorrências em 105 tokens) e o 1097 (8 ocorrências de `ablat` e 7 de
`materi`). O BM25, graças à saturação de frequência controlada por k1 e à
normalização por tamanho, não deixa essa repetição dominar o score e
recupera o 274 em 1º e o 82 em 4º.

**Consulta 173** (AP 1,000 no BM25 contra 0,583 no Vetorial) tem dois
documentos relevantes, 367 e 451, sobre o método de Lyapunov. O Modelo
Vetorial insere na 1ª posição o documento 532, julgado não relevante
(grau -1), que menciona estabilidade e o segundo método de Lyapunov por
coincidência lexical. O BM25 também recupera o 532, mas só em 3º, e coloca
os dois relevantes em 1º e 2º, obtendo AP perfeito. Ou seja, ele não é
imune ao mesmo distrator, apenas o pondera menos.

In [2]:
for qid in ["167", "173"]:
    show_top_n(qid)

Consulta 167: exact solution methods for calculating the ablative mass loss of a material ablating at high temperatures in a hypersonic flight environment .
AP -> BM25: 0.750 | Modelo Vetorial: 0.091
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,274,2,SIM,analysis of quartz and teflon shields for a pa...
2,1279,não julgado,não,sublimation in a hypersonic environment .
3,553,não julgado,não,ablation of glassy materials around blunt bodi...
4,82,3,SIM,theoretical investigation of the ablation of a...
5,1098,não julgado,não,an experimental investigation of ablating mate...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,553,não julgado,não,ablation of glassy materials around blunt bodi...
2,1279,não julgado,não,sublimation in a hypersonic environment .
3,1099,não julgado,não,a theoretical study of stagnation point ablati...
4,1100,não julgado,não,an analytical investigation of ablation .
5,1097,não julgado,não,experimental ablation cooling .



Consulta 173: references on lyapunov's method on the stability of linear differential equations with periodic coefficients .
AP -> BM25: 1.000 | Modelo Vetorial: 0.583
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,367,1,SIM,control system and analysis and design via the...
2,451,1,SIM,liapunov's methods in automatic control theory .
3,532,-1,não,pitch-yaw stability of a missile oscillating i...
4,917,não julgado,não,a method of calculating the short period longi...
5,767,não julgado,não,mathematical techniques applying to the therma...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,532,-1,não,pitch-yaw stability of a missile oscillating i...
2,367,1,SIM,control system and analysis and design via the...
3,451,1,SIM,liapunov's methods in automatic control theory .
4,917,não julgado,não,a method of calculating the short period longi...
5,1073,não julgado,não,a practical method for numerical evaluation of...


## 2) Consultas em que o Modelo Vetorial é claramente superior

**Consulta 17** (AP 0,508 no Vetorial contra 0,090 no BM25) é o caso mais
extremo na direção oposta. Os termos que definem a pergunta são
`transvers` e `potenti`; o resto (`problem`, `flow`, `bodi`, `dimension`)
é vocabulário genérico da coleção. O Vetorial acerta em 1º o documento
106, relevante de grau 2 e com apenas 43 tokens, exatamente por casar os
dois termos específicos. O BM25 promove o documento 1108 (154 tokens, não
julgado), que acumula os termos genéricos e **não contém nenhum dos dois
termos definidores**, e não traz nenhum relevante ao Top 5.

**Consulta 145** (AP 0,583 no Vetorial contra 0,279 no BM25) repete o
padrão com mais documentos relevantes em jogo. O Vetorial coloca em 1º o
documento 1045, de grau 3 e apenas **19 tokens** ("the bending strength of
pressurized cylinders"), em 2º o 839, de grau 2, e em 4º o 763, de grau 2 —
três relevantes nas quatro primeiras posições. O BM25 prefere o documento
1051, não julgado e com 140 tokens, que toca 10 termos distintos da
consulta mas nenhum deles de forma decisiva, empurrando os relevantes para
3º e 4º.

Nos dois casos o mecanismo é o mesmo: o BM25 recompensa **cobertura ampla e
rasa** num documento longo, enquanto o cosseno do Vetorial recompensa
**concentração** num documento curto.

In [3]:
for qid in ["17", "145"]:
    show_top_n(qid)

Consulta 17: can the three-dimensional problem of a transverse potential flow about a body of revolution be reduced to a two-dimensional problem .
AP -> BM25: 0.090 | Modelo Vetorial: 0.508
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,1108,não julgado,não,a study of second-order supersonic flow theory .
2,1281,não julgado,não,turbulent heat transfer on blunt-nosed bodies ...
3,700,não julgado,não,two and three-dimensional unsteady lift proble...
4,336,não julgado,não,simplified laminar boundary layer calculations...
5,1301,não julgado,não,compressible boundary layers on bodies of revo...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,106,2,SIM,the transverse potential flow past a body of r...
2,1281,não julgado,não,turbulent heat transfer on blunt-nosed bodies ...
3,700,não julgado,não,two and three-dimensional unsteady lift proble...
4,1301,não julgado,não,compressible boundary layers on bodies of revo...
5,498,-1,não,calculation of potential flow about bodies of ...



Consulta 145: what are the best experimental data and classical small deflection theory analyses available for pressurized cylinders in bending .
AP -> BM25: 0.279 | Modelo Vetorial: 0.583
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,1051,não julgado,não,the stability of thin-walled unstiffened circu...
2,1126,não julgado,não,an engineer's conceptual approach to the buckl...
3,839,2,SIM,the bending stability of thin-- walled unstiff...
4,1045,3,SIM,the bending strength of pressurized cylinders .
5,1118,não julgado,não,elastic stability of orthotropic shells .


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,1045,3,SIM,the bending strength of pressurized cylinders .
2,839,2,SIM,the bending stability of thin-- walled unstiff...
3,1051,não julgado,não,the stability of thin-walled unstiffened circu...
4,763,2,SIM,effects of internal pressure on the buckling o...
5,1046,não julgado,não,the bending strength of pressurized cylinders .


## 3) Consultas em que ambos os modelos falham

**Consulta 31** pergunta que tamanho de placa de extremidade pode ser usado
com segurança para simular condições de escoamento bidimensional. Possui um
único documento relevante, o 776, de grau 4, que nenhum dos dois modelos
chega perto do Top 10: ele fica em 1087º no BM25 e 1124º no Vetorial, de
1400. Pior, o documento 751, explicitamente não relevante (grau -1), é
colocado em 1º **por ambos**: ele compartilha 7 dos 16 termos da consulta
(`end` 6 vezes, `plate` 3, `dimension` 3, `flow` 3, `bluff` 2) mas trata do
problema oposto — como *evitar* o efeito tridimensional, em vez de
dimensionar a placa que o simula.

**Consulta 22** pergunta se alguém mais descobriu que o atrito de pele
turbulento não é muito sensível à variação da viscosidade com a
temperatura. O único documento relevante, o 68, de grau 1, fica em 606º em
ambos os rankings. Seu título, "some aspects of air-helium simulation and
hypersonic approximations", praticamente não compartilha termos de conteúdo
com a consulta: a relevância depende de uma ligação conceitual discutida no
corpo do texto.

Em ambos os casos o problema não está em qual modelo foi escolhido. É o
clássico problema de vocabulário, de sinonímia e paráfrase: quando a
consulta e o documento relevante descrevem o mesmo conceito com palavras de
superfície muito diferentes, nenhum modelo de correspondência exata de
termos os aproxima. Isso é retomado em detalhe no Caso 9.

In [4]:
for qid in ["31", "22"]:
    show_top_n(qid)

# Trava os fatos citados nas tres analises acima.
assert bm25_rankings["167"][0] == "274" and bm25_rankings["167"][3] == "82"
assert vsm_rankings["173"][0] == "532" and bm25_rankings["173"][:2] == ["367", "451"]
assert vsm_rankings["17"][0] == "106" and bm25_rankings["17"][0] == "1108"
assert vsm_rankings["145"][:2] == ["1045", "839"] and bm25_rankings["145"][0] == "1051"
assert bm25_rankings["31"][0] == "751" and vsm_rankings["31"][0] == "751"
assert bm25_rankings["31"].index("776") + 1 == 1087
assert vsm_rankings["31"].index("776") + 1 == 1124
assert bm25_rankings["22"].index("68") + 1 == 606 == vsm_rankings["22"].index("68") + 1
print("OK: todas as posicoes citadas nas analises conferem com os rankings.")

Consulta 31: what size of end plate can be safely used to simulate two-dimensional flow conditions over a bluff cylindrical body of finite aspect ratio .
AP -> BM25: 0.001 | Modelo Vetorial: 0.001
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,751,-1,não,a note on the use of end plates to prevent thr...
2,1153,não julgado,não,a study of the simulation of flow with free st...
3,1209,não julgado,não,aerodynamic processes in the downwash-impingem...
4,1245,não julgado,não,some aspects of nonequilibrium flows .
5,228,não julgado,não,navier-stokes solutions at large distances fro...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,751,-1,não,a note on the use of end plates to prevent thr...
2,68,não julgado,não,some aspects of air-helium simulation and hype...
3,247,não julgado,não,the calculation of the pressure distribution o...
4,916,não julgado,não,the flow around oscillating low aspect ratio w...
5,918,não julgado,não,on the low aspect ratio oscillating rectangula...



Consulta 22: did anyone else discover that the turbulent skin friction is not over sensitive to the nature of the variation of the viscosity with temperature .
AP -> BM25: 0.002 | Modelo Vetorial: 0.002
-- Top-5 BM25 --


,doc_id,grau,relevante?,titulo
rank,,,,
1,125,não julgado,não,measurements of skin friction of the compressi...
2,560,não julgado,não,a theoretical study of the effect of upstream ...
3,413,não julgado,não,turbulent skin friction at high mach numbers a...
4,50,não julgado,não,investigation of laminar boundary layer in com...
5,81,não julgado,não,compressible laminar flow and heat transfer ab...


-- Top-5 Modelo Vetorial --


,doc_id,grau,relevante?,titulo
rank,,,,
1,125,não julgado,não,measurements of skin friction of the compressi...
2,413,não julgado,não,turbulent skin friction at high mach numbers a...
3,165,não julgado,não,skin-friction measurements in incompressible f...
4,254,não julgado,não,boundary layers with suction and injection . a...
5,560,não julgado,não,a theoretical study of the effect of upstream ...



OK: todas as posicoes citadas nas analises conferem com os rankings.
